# Week 58, Day 5: Interpreting ensembles: importance and partial dependence

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w058/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(585)
names = ["speed", "direction", "reward", "noise_cont", "noise_bin"]

def session(n, g):
    speed = g.uniform(0, 1, n)                         # normalised running speed, many distinct values
    direction = g.integers(0, 4, n)                    # one of four stimulus directions
    reward = g.integers(0, 2, n)                       # rewarded or not
    noise_cont = g.uniform(0, 1, n)                    # irrelevant, continuous
    noise_bin = g.integers(0, 2, n)                    # irrelevant, binary
    rate = 5 + 4 * speed + 3 * np.cos(np.pi * direction / 2) + 6 * speed * reward
    X = np.column_stack([speed, direction, reward, noise_cont, noise_bin]).astype(float)
    return X, rate + 1.5 * g.normal(size=n), rate

def best_split(X, y, feats, min_leaf):
    n, best = len(y), (None, 0.0, 0.0)
    for j in feats:
        o = np.argsort(X[:, j]); xs, ys = X[o, j], y[o]
        cs = np.cumsum(ys)[:-1]; nl = np.arange(1, n)
        gain = cs ** 2 / nl + (ys.sum() - cs) ** 2 / (n - nl) - ys.sum() ** 2 / n
        ok = (xs[1:] > xs[:-1]) & (nl >= min_leaf) & (n - nl >= min_leaf)
        gain = np.where(ok, gain, -np.inf); k = int(np.argmax(gain))
        if gain[k] > best[2]: best = (j, (xs[k] + xs[k + 1]) / 2, gain[k])
    return best

def grow(X, y, g, mtry, min_leaf, imp_acc):
    if len(y) < 2 * min_leaf: return ("leaf", y.mean())
    feats = g.choice(X.shape[1], mtry, replace=False)
    j, t, gain = best_split(X, y, feats, min_leaf)
    if j is None: return ("leaf", y.mean())
    imp_acc[j] += gain                                 # reduction in squared error from this split
    m = X[:, j] <= t
    return ("node", j, t, grow(X[m], y[m], g, mtry, min_leaf, imp_acc), grow(X[~m], y[~m], g, mtry, min_leaf, imp_acc))

def predict(tree, X):
    if tree[0] == "leaf": return np.full(len(X), tree[1])
    _, j, t, L, R = tree; m = X[:, j] <= t; out = np.empty(len(X))
    out[m], out[~m] = predict(L, X[m]), predict(R, X[~m]); return out

def forest(X, y, g, n_trees=40, mtry=2, min_leaf=3):
    imp_acc, trees = np.zeros(X.shape[1]), []
    for b in range(n_trees):
        idx = g.integers(0, len(y), len(y))
        trees.append(grow(X[idx], y[idx], g, mtry, min_leaf, imp_acc))
    return trees, imp_acc / imp_acc.sum()

fpredict = lambda trees, X: np.mean([predict(t, X) for t in trees], axis=0)
r2 = lambda y_, p: 1 - np.mean((y_ - p) ** 2) / np.var(y_)

X, y, _ = session(400, r); X_te, y_te, rate_te = session(2000, r)
trees, imp_impurity = forest(X, y, np.random.default_rng(1))
p_te = fpredict(trees, X_te); r2_te = r2(y_te, p_te)
print(f"forest test R squared {r2_te:.3f}; ceiling set by the noise {r2(y_te, rate_te):.3f}")

### Exercise: Impurity against permutation importance

Write permutation_importance(trees, X, y, g, repeats): for each feature, shuffle that column of X (on a copy), predict, and record the rise in mean squared error, averaged over repeats. Normalise it by the variance of y. Compare it with the impurity importance for the five features.

In [ ]:
def permutation_importance(trees, X, y, g, repeats=3):
    base = np.mean((y - fpredict(trees, X)) ** 2)
    out = np.zeros(X.shape[1])
    for j in range(X.shape[1]):
        for _ in range(repeats):
            Xp = X.copy()
            Xp[:, j] = ...
            out[j] += (np.mean((y - fpredict(trees, Xp)) ** 2) - base) / repeats
    return out / np.var(y)

imp_perm = permutation_importance(trees, X_te[:600], y_te[:600], np.random.default_rng(2))
print(f"{'feature':>11}  impurity  permutation")
for k, nm in enumerate(names):
    print(f"{nm:>11}  {imp_impurity[k]:8.3f}  {imp_perm[k]:11.3f}")

In [ ]:
import numpy as np
def test_perm():
    "permutation importance is a length-5 array, with speed the most important"
    assert len(imp_perm) == 5 and int(np.argmax(imp_perm)) == 0
def test_bias():
    "the irrelevant continuous feature gets clear impurity importance but almost no permutation importance"
    assert imp_impurity[3] > 3 * imp_impurity[4] and imp_impurity[3] > 0.04 and imp_perm[3] < 0.02

test_perm()
test_bias()
print("All checks passed")

### Exercise: A second copy of speed

Add a sixth feature, speed plus a little noise (standard deviation 0.03), to the training and test data. Refit the forest with mtry 3 and recompute permutation importance. Compare the importance of speed with and without the copy.

In [ ]:
gc = np.random.default_rng(3)
X2 = np.column_stack([X, X[:, 0] + 0.03 * gc.normal(size=len(X))])
X2_te = ...
trees2, _ = forest(X2, y, np.random.default_rng(1), mtry=3)
imp_copy = permutation_importance(trees2, X2_te[:600], y_te[:600], np.random.default_rng(2))
print(f"speed alone: {imp_perm[0]:.3f}; with a copy present: speed {imp_copy[0]:.3f}, copy {imp_copy[5]:.3f}")
print(f"test R squared with the copy: {r2(y_te, fpredict(trees2, X2_te)):.3f}")

In [ ]:
import numpy as np
def test_split():
    "adding a near copy of speed lowers the permutation importance of the original by at least a third"
    assert imp_copy[0] < (2 / 3) * imp_perm[0]
def test_share():
    "the copy itself picks up a clear share of importance"
    assert imp_copy[5] > 0.05

test_split()
test_share()
print("All checks passed")

### Exercise: How firing depends on speed

Write ice(trees, X, j, grid), returning an array of predictions with one row per trial and one column per grid value, with feature j set to that value. The partial dependence is its column mean. Then compare the average ICE slope (last column minus first) on rewarded and unrewarded trials.

In [ ]:
def ice(trees, X, j, grid):
    out = np.empty((len(X), len(grid)))
    for k, v in enumerate(grid):
        Xv = X.copy()
        Xv[:, j] = v
        out[:, k] = ...
    return out

grid = np.linspace(0.05, 0.95, 10)
curves = ice(trees, X_te[:300], 0, grid)
pd_speed = curves.mean(0)
rew = X_te[:300, 2] == 1
slope_rew = np.mean(curves[rew, -1] - curves[rew, 0])
slope_unrew = np.mean(curves[~rew, -1] - curves[~rew, 0])
print("partial dependence on speed:", np.round(pd_speed, 1))
print(f"ICE rise across the grid: rewarded {slope_rew:.1f}, unrewarded {slope_unrew:.1f} (truth 9.0 and 3.6)")
fig, ax = plt.subplots(figsize=(5, 3))
for c, rw in zip(curves[:60], rew[:60]): ax.plot(grid, c, color="C3" if rw else "C0", alpha=0.3, lw=0.8)
ax.plot(grid, pd_speed, "k", lw=2.5); ax.set_xlabel("speed"); ax.set_ylabel("predicted rate")
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
def test_pd():
    "the partial dependence on speed rises across the grid"
    assert pd_speed[-1] - pd_speed[0] > 3
def test_ice():
    "ICE slopes are steeper on rewarded trials than on unrewarded ones"
    assert slope_rew > slope_unrew + 3

test_pd()
test_ice()
print("All checks passed")